# Project 3 - SQL Data Analysis

Same order dataset as project 2 (1200 e-commerce orders), loaded into a SQLite
database and queried directly with SQL instead of pandas. Covers SELECT,
WHERE, ORDER BY, GROUP BY, HAVING, and the basic aggregations (COUNT, SUM,
AVG).


## 1. Imports and connecting to the database

In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../database/orders.db")
pd.set_option("display.max_columns", None)

## 2. Building the database

The raw csv gets loaded into a single `orders` table. This only needs to run
once - if the .db file already exists this just refreshes it.

In [2]:
df = pd.read_csv("../data/orders.csv", parse_dates=["Date"])
df.to_sql("orders", conn, if_exists="replace", index=False)
print(df.shape)

(1200, 14)


In [3]:
pd.read_sql("SELECT * FROM orders LIMIT 5;", conn)

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04 00:00:00,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23 00:00:00,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27 00:00:00,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15 00:00:00,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08 00:00:00,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


## 3. Basic SELECT

Picking specific columns instead of pulling everything with `*`.

In [4]:
query = """
SELECT OrderID, Product, Quantity, UnitPrice, TotalPrice
FROM orders
LIMIT 10;
"""
pd.read_sql(query, conn)

,OrderID,Product,Quantity,UnitPrice,TotalPrice
0,ORD200000,Monitor,5,570.62,2853.10
1,ORD200001,Phone,2,151.35,302.70
2,ORD200002,Tablet,5,550.68,2753.40
3,ORD200003,Chair,1,273.19,273.19
4,ORD200004,Printer,4,626.01,2504.04
5,ORD200005,Phone,2,245.86,491.72
6,ORD200006,Laptop,1,664.42,664.42
7,ORD200007,Monitor,5,149.55,747.75
8,ORD200008,Phone,2,134.28,268.56
9,ORD200009,Desk,4,509.38,2037.52


## 4. WHERE - filtering rows

In [5]:
query = """
SELECT OrderID, Product, TotalPrice, OrderStatus
FROM orders
WHERE OrderStatus = 'Delivered'
LIMIT 10;
"""
pd.read_sql(query, conn)

,OrderID,Product,TotalPrice,OrderStatus
0,ORD200004,Printer,2504.04,Delivered
1,ORD200015,Printer,473.96,Delivered
2,ORD200021,Monitor,1371.80,Delivered
3,ORD200024,Laptop,461.90,Delivered
4,ORD200025,Monitor,399.98,Delivered
5,ORD200026,Laptop,297.75,Delivered
6,ORD200029,Desk,1270.59,Delivered
7,ORD200032,Tablet,2683.60,Delivered
8,ORD200038,Laptop,245.89,Delivered
9,ORD200039,Printer,769.38,Delivered


In [6]:
query = """
SELECT OrderID, Product, TotalPrice
FROM orders
WHERE TotalPrice > 2000
ORDER BY TotalPrice DESC;
"""
pd.read_sql(query, conn)

,OrderID,Product,TotalPrice
0,ORD200789,Tablet,3456.40
1,ORD201122,Monitor,3390.95
2,ORD200632,Laptop,3390.80
3,ORD200469,Chair,3384.90
4,ORD200328,Tablet,3370.20
...,...,...,...
175,ORD201035,Laptop,2024.44
176,ORD200181,Printer,2021.95
177,ORD200915,Laptop,2011.96
178,ORD200979,Monitor,2007.18


In [7]:
query = """
SELECT OrderID, Product, Quantity, UnitPrice, TotalPrice
FROM orders
WHERE Product = 'Laptop' AND TotalPrice > 1000
ORDER BY TotalPrice DESC;
"""
pd.read_sql(query, conn)

,OrderID,Product,Quantity,UnitPrice,TotalPrice
0,ORD200632,Laptop,5,678.16,3390.80
1,ORD200326,Laptop,5,670.48,3352.40
2,ORD200463,Laptop,5,662.78,3313.90
3,ORD200367,Laptop,5,658.77,3293.85
4,ORD200540,Laptop,5,648.65,3243.25
...,...,...,...,...,...
72,ORD201092,Laptop,5,209.20,1046.00
73,ORD200939,Laptop,2,521.40,1042.80
74,ORD200284,Laptop,5,208.17,1040.85
75,ORD200373,Laptop,3,341.15,1023.45


Pattern matching with LIKE:

In [8]:
query = """
SELECT OrderID, ShippingAddress
FROM orders
WHERE ShippingAddress LIKE '%Main St%'
LIMIT 10;
"""
pd.read_sql(query, conn)

,OrderID,ShippingAddress
0,ORD200000,928 Main St
1,ORD200001,823 Main St
2,ORD200002,512 Main St
3,ORD200003,275 Main St
4,ORD200004,668 Main St
5,ORD200005,934 Main St
6,ORD200006,986 Main St
7,ORD200007,706 Main St
8,ORD200008,904 Main St
9,ORD200009,102 Main St


## 5. ORDER BY - sorting results

In [9]:
query = """
SELECT OrderID, Product, TotalPrice
FROM orders
ORDER BY TotalPrice DESC
LIMIT 10;
"""
pd.read_sql(query, conn)

,OrderID,Product,TotalPrice
0,ORD200789,Tablet,3456.40
1,ORD201122,Monitor,3390.95
2,ORD200632,Laptop,3390.80
3,ORD200469,Chair,3384.90
4,ORD200328,Tablet,3370.20
5,ORD200107,Printer,3353.75
6,ORD200326,Laptop,3352.40
7,ORD201065,Printer,3334.00
8,ORD201031,Phone,3322.55
9,ORD200463,Laptop,3313.90


In [10]:
query = """
SELECT OrderID, Product, TotalPrice
FROM orders
ORDER BY TotalPrice ASC
LIMIT 10;
"""
pd.read_sql(query, conn)

,OrderID,Product,TotalPrice
0,ORD201161,Phone,11.39
1,ORD200863,Phone,14.06
2,ORD200240,Tablet,17.24
3,ORD200542,Tablet,17.98
4,ORD200336,Laptop,18.20
5,ORD200776,Tablet,21.19
6,ORD201025,Chair,23.53
7,ORD200690,Monitor,24.48
8,ORD200926,Desk,26.95
9,ORD200473,Chair,29.86


## 6. GROUP BY with aggregations

COUNT, SUM, and AVG applied per category.

In [11]:
query = """
SELECT Product, COUNT(*) AS order_count
FROM orders
GROUP BY Product
ORDER BY order_count DESC;
"""
pd.read_sql(query, conn)

,Product,order_count
0,Printer,181
1,Tablet,179
2,Chair,178
3,Laptop,173
4,Desk,170
5,Monitor,163
6,Phone,156


In [12]:
query = """
SELECT Product, SUM(TotalPrice) AS total_revenue
FROM orders
GROUP BY Product
ORDER BY total_revenue DESC;
"""
pd.read_sql(query, conn)

,Product,total_revenue
0,Chair,195620.11
1,Printer,195612.61
2,Laptop,192126.56
3,Tablet,186568.95
4,Monitor,175651.41
5,Desk,167459.93
6,Phone,151722.39


In [13]:
query = """
SELECT Product, ROUND(AVG(TotalPrice), 2) AS avg_order_value
FROM orders
GROUP BY Product
ORDER BY avg_order_value DESC;
"""
pd.read_sql(query, conn)

,Product,avg_order_value
0,Laptop,1110.56
1,Chair,1098.99
2,Printer,1080.73
3,Monitor,1077.62
4,Tablet,1042.28
5,Desk,985.06
6,Phone,972.58


In [14]:
query = """
SELECT OrderStatus, COUNT(*) AS order_count
FROM orders
GROUP BY OrderStatus
ORDER BY order_count DESC;
"""
pd.read_sql(query, conn)

,OrderStatus,order_count
0,Cancelled,250
1,Returned,247
2,Pending,237
3,Shipped,235
4,Delivered,231


In [15]:
query = """
SELECT PaymentMethod, COUNT(*) AS order_count, SUM(TotalPrice) AS total_revenue
FROM orders
GROUP BY PaymentMethod
ORDER BY total_revenue DESC;
"""
pd.read_sql(query, conn)

,PaymentMethod,order_count,total_revenue
0,Credit Card,234,263847.63
1,Online,258,262442.94
2,Cash,246,259786.29
3,Gift Card,230,246323.92
4,Debit Card,232,232361.18


In [16]:
query = """
SELECT ReferralSource, COUNT(*) AS order_count, SUM(TotalPrice) AS total_revenue
FROM orders
GROUP BY ReferralSource
ORDER BY total_revenue DESC;
"""
pd.read_sql(query, conn)

,ReferralSource,order_count,total_revenue
0,Instagram,259,275285.45
1,Email,250,261808.55
2,Google,241,250441.48
3,Facebook,228,250410.90
4,Referral,222,226815.58


## 7. HAVING - filtering grouped results

WHERE filters rows before grouping, HAVING filters the groups themselves
after the aggregation is done.

In [17]:
query = """
SELECT Product, COUNT(*) AS order_count
FROM orders
GROUP BY Product
HAVING COUNT(*) > 170
ORDER BY order_count DESC;
"""
pd.read_sql(query, conn)

,Product,order_count
0,Printer,181
1,Tablet,179
2,Chair,178
3,Laptop,173


In [18]:
query = """
SELECT PaymentMethod, ROUND(AVG(TotalPrice), 2) AS avg_order_value
FROM orders
GROUP BY PaymentMethod
HAVING AVG(TotalPrice) > 1000
ORDER BY avg_order_value DESC;
"""
pd.read_sql(query, conn)

,PaymentMethod,avg_order_value
0,Credit Card,1127.55
1,Gift Card,1070.97
2,Cash,1056.04
3,Online,1017.22
4,Debit Card,1001.56


## 8. Grouping by month

In [19]:
query = """
SELECT strftime('%Y-%m', Date) AS order_month, COUNT(*) AS order_count, SUM(TotalPrice) AS total_revenue
FROM orders
GROUP BY order_month
ORDER BY order_month;
"""
monthly = pd.read_sql(query, conn)
monthly

,order_month,order_count,total_revenue
0,2023-01,47,56685.75
1,2023-02,37,40117.66
2,2023-03,43,48609.37
3,2023-04,31,27751.71
4,2023-05,49,63836.84
5,2023-06,45,49500.19
6,2023-07,44,42820.66
7,2023-08,51,54352.14
8,2023-09,29,29526.67
9,2023-10,47,52607.85


## 9. Coupon usage

In [20]:
query = """
SELECT COALESCE(CouponCode, 'NONE') AS coupon_used, COUNT(*) AS order_count
FROM orders
GROUP BY coupon_used
ORDER BY order_count DESC;
"""
pd.read_sql(query, conn)

,coupon_used,order_count
0,FREESHIP,313
1,NONE,309
2,WINTER15,292
3,SAVE10,286


## 10. Percentage contribution

Using a subquery to get each product's share of total revenue.

In [21]:
query = """
SELECT Product,
       SUM(TotalPrice) AS total_revenue,
       ROUND(SUM(TotalPrice) * 100.0 / (SELECT SUM(TotalPrice) FROM orders), 2) AS pct_of_total_revenue
FROM orders
GROUP BY Product
ORDER BY pct_of_total_revenue DESC;
"""
pd.read_sql(query, conn)

,Product,total_revenue,pct_of_total_revenue
0,Printer,195612.61,15.47
1,Chair,195620.11,15.47
2,Laptop,192126.56,15.19
3,Tablet,186568.95,14.75
4,Monitor,175651.41,13.89
5,Desk,167459.93,13.24
6,Phone,151722.39,12.00


## 11. Top orders and repeat customers

In [22]:
query = """
SELECT OrderID, Product, Quantity, UnitPrice, TotalPrice, OrderStatus
FROM orders
ORDER BY TotalPrice DESC
LIMIT 5;
"""
pd.read_sql(query, conn)

,OrderID,Product,Quantity,UnitPrice,TotalPrice,OrderStatus
0,ORD200789,Tablet,5,691.28,3456.40,Delivered
1,ORD201122,Monitor,5,678.19,3390.95,Returned
2,ORD200632,Laptop,5,678.16,3390.80,Delivered
3,ORD200469,Chair,5,676.98,3384.90,Cancelled
4,ORD200328,Tablet,5,674.04,3370.20,Cancelled


In [23]:
query = """
SELECT CustomerID, COUNT(*) AS order_count
FROM orders
GROUP BY CustomerID
HAVING COUNT(*) > 1
ORDER BY order_count DESC;
"""
pd.read_sql(query, conn)

,CustomerID,order_count
0,C98474,2
1,C97593,2
2,C94569,2
3,C91155,2
4,C70659,2
5,C56969,2
6,C46651,2
7,C38840,2
8,C35852,2
9,C21191,2


## 12. Cancelled and returned share

In [24]:
query = """
SELECT
    SUM(CASE WHEN OrderStatus IN ('Cancelled', 'Returned') THEN 1 ELSE 0 END) AS cancelled_returned_count,
    COUNT(*) AS total_orders,
    ROUND(SUM(CASE WHEN OrderStatus IN ('Cancelled', 'Returned') THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS cancelled_returned_pct
FROM orders;
"""
pd.read_sql(query, conn)

,cancelled_returned_count,total_orders,cancelled_returned_pct
0,497,1200,41.42


## 13. Summary

- Printer and Tablet have the highest order counts, but that's not the same
  as highest revenue - UnitPrice varies a lot between products
- Cancelled and Returned orders together make up about 41% of all orders
- Payment method doesn't change average order value by much, they're all in
  a similar range
- Percentage contribution query shows revenue is fairly evenly spread across
  the 7 products, nothing dominates
- 655 different shipping addresses show up across 1200 orders, so quite a
  few repeat addresses


In [25]:
conn.close()